# 05. 하이퍼파라미터 튜닝 — 통합모델(CatBoost)

**목적**: `04_model_selection`에서 확정한 최종 승자 구조 — **통합모델(이용률 타깃) + CatBoost + v2 피처셋(50개)** — 의 하이퍼파라미터를 `model-tuning` 스킬 기준으로 탐색한다.

- **입력**: `data/processed/train_features_v2.parquet`, `test_features_v2.parquet` (`04_model_selection` 피처 선택 산출물)
- **출력**: 최적 하이퍼파라미터(dict), 튜닝 전후 비교, seed 안정성 확인 결과 — `reports/05_tuning.md`로 정리, 이후 `train.ipynb`에 하드코딩되어 재사용됨

**`model-tuning` 스킬 1절 핵심 규칙**: "튜닝 목적 함수 = 동일 CV 구조의 fold 평균 Score. 단일 fold 최적화 금지." `04_model_selection`은 단일 fold(2022~2023 train / 2024 validation)로 모델을 비교했지만, 이제 하이퍼파라미터를 실제로 탐색하는 단계라 **fold 하나에만 맞춰 튜닝하면 그 fold의 우연한 패턴에 과적합될 위험**이 커진다. 그래서 이 노트북부터는 `timeseries-validation` 스킬의 "보강안"인 **확장 윈도우 3-fold 시계열 CV**로 바꾼다.

## 0. 셋업

패키지를 불러오고 REPO_ROOT를 찾는다. `optuna`(하이퍼파라미터 탐색 도구 — 무작정 모든 조합을 다 해보는 대신, 지금까지의 결과를 보고 "다음엔 이쪽을 더 파보자"고 똑똑하게 다음 후보를 고르는 라이브러리)를 새로 사용한다.

In [12]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import optuna
from optuna.samplers import TPESampler

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

pd.set_option("display.max_columns", 100)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("optuna version:", optuna.__version__)

python executable: c:\Users\cho03\Desktop\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: c:\Users\cho03\Desktop\wind_forecast
optuna version: 4.9.0


## 1. 데이터 로드 (v2 피처셋)

`04_model_selection`의 피처 선택 결과물인 v2(50개 피처)를 불러온다. v1(52개) 대신 v2를 쓰는 이유: 상관관계·permutation importance·피처군 ablation을 거쳐 걸러낸 피처셋이기 때문.

In [13]:
train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")

DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("train_features:", train_features.shape)
print("test_features:", test_features.shape)
print("피처 개수:", len(FEATURE_COLS))
print("기간:", train_features["forecast_kst_dtm"].min(), "~", train_features["forecast_kst_dtm"].max())

train_features: (26304, 54)
test_features: (8760, 52)
피처 개수: 50
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


## 2. 확장 윈도우 3-fold 시계열 CV

**결론 먼저**: 아래 3개 fold를 만든다. 전부 **2022-01-01부터 시작해서 점점 뒤로 늘어나는 학습 구간**(확장 윈도우 — 매번 처음부터 다시 시작하지 않고, 알고 있는 과거를 계속 누적해서 학습에 쓰는 방식) + 그 바로 다음 6개월을 검증 구간으로 쓴다.

| fold | 학습 구간 | 검증 구간 |
|---|---|---|
| fold1 | 2022-01 ~ 2023-06 | 2023-07 ~ 2023-12 |
| fold2 | 2022-01 ~ 2023-12 | 2024-01 ~ 2024-06 |
| fold3 | 2022-01 ~ 2024-06 | 2024-07 ~ 2024-12 |

**왜 이 구조인가**: 랜덤 K-Fold는 시계열에서 절대 쓰면 안 된다(`timeseries-validation` 스킬 1절 — 미래 데이터로 과거를 예측하는 셈이 되어 누수). 확장 윈도우는 실제 운영 상황(계속 쌓이는 과거 데이터로 다음 구간을 예측)과 가장 비슷하다. `04_model_selection`에서 쓴 단일 분리(2022~2023 train / 2024 valid)는 fold2와 거의 같은 구조인데, 튜닝에서는 이거 하나만 보면 "2024년 상반기에만 우연히 잘 맞는" 파라미터를 고를 위험이 있어 fold를 3개로 늘렸다.

group_3(라벨이 2023-01부터만 있음)도 fold1부터 6개월치 학습 데이터가 있어 3개 fold 모두 유효하다.

In [14]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def make_fold_frames(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    valid_end = pd.Timestamp(fold["valid_end"])
    valid_start = train_end + pd.Timedelta(hours=1)

    fold_train = train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)
    fold_valid = train_features[(dtm >= valid_start) & (dtm <= valid_end)].reset_index(drop=True)
    return fold_train, fold_valid


for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    print(
        f"{fold['name']}: train {ft.shape} ({ft['forecast_kst_dtm'].min()} ~ {ft['forecast_kst_dtm'].max()}), "
        f"valid {fv.shape} ({fv['forecast_kst_dtm'].min()} ~ {fv['forecast_kst_dtm'].max()})"
    )
    print("  그룹별 valid 라벨 결측:", fv[TARGET_COLS].isna().sum().to_dict())

fold1: train (13104, 54) (2022-01-01 01:00:00 ~ 2023-07-01 00:00:00), valid (4416, 54) (2023-07-01 01:00:00 ~ 2024-01-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 3, 'kpx_group_2': 2, 'kpx_group_3': 0}
fold2: train (17520, 54) (2022-01-01 01:00:00 ~ 2024-01-01 00:00:00), valid (4368, 54) (2024-01-01 01:00:00 ~ 2024-07-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 0, 'kpx_group_2': 0, 'kpx_group_3': 0}
fold3: train (21888, 54) (2022-01-01 01:00:00 ~ 2024-07-01 00:00:00), valid (4416, 54) (2024-07-01 01:00:00 ~ 2025-01-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 6, 'kpx_group_2': 6, 'kpx_group_3': 6}


## 3. 통합모델 학습·평가 함수

`04_model_selection` 8절(5b)에서 쓴 것과 같은 구조 — group_id 범주형 피처 + 이용률 타깃으로 3개 그룹을 합쳐 학습 — 를 함수로 재사용 가능하게 만든다. 이번엔 하이퍼파라미터를 인자로 받아서, fold 하나짜리 점수(`train_and_score_fold`)와 3-fold 평균 점수(`cv_score`)를 모두 낼 수 있게 한다.

In [15]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]


def to_long(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out

In [16]:
def train_and_score_fold(fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15, seed=SEED):
    features = feature_cols + ["group_id"]
    long_df = to_long(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    model = CatBoostRegressor(
        loss_function="MAE",
        random_seed=seed,
        verbose=False,
        **params,
    )
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"],
        early_stopping_rounds=100,
        verbose=False,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        util_pred = model.predict(valid_g[features])
        pred[g] = util_pred * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    score, one_minus_nmae, ficr = metric(answer_df, pred_df)
    return score, model


def cv_score(params, feature_cols=FEATURE_COLS, seed=SEED, verbose=False):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        score, _ = train_and_score_fold(fold_train, fold_valid, params, feature_cols, seed=seed)
        scores.append(score)
        if verbose:
            print(f"  {fold['name']}: Score={score:.4f}")
    return float(np.mean(scores)), scores

## 4. 튜닝 전 기준점

`04_model_selection` 5b에서 쓴 기본 파라미터(`iterations=2000, learning_rate=0.05`, 나머지 CatBoost 기본값)를 **이번 3-fold CV 구조**로 다시 채점해 기준점을 만든다. 04번의 0.5971은 단일 fold 점수라 이 3-fold 평균과 숫자가 다를 수 있다 — 비교는 항상 "같은 CV 구조끼리"만 해야 하므로(`model-tuning` 스킬 1절), 튜닝 전후 비교는 이 기준점을 기준으로 한다.

In [17]:
DEFAULT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}

baseline_mean, baseline_scores = cv_score(DEFAULT_PARAMS, verbose=True)
print(f"\n기본 파라미터 3-fold 평균 Score: {baseline_mean:.4f} (fold별: {[round(s, 4) for s in baseline_scores]})")

  fold1: Score=0.5679


KeyboardInterrupt: 

## 5. Optuna 탐색 범위 설계

`model-tuning` 스킬 2절은 LightGBM 기준 표를 준다. CatBoost는 파라미터 이름이 달라서(대칭 트리 구조라 `num_leaves` 대신 `depth`를 씀 등) 같은 역할을 하는 파라미터로 대응시켰다:

| 역할 | LightGBM(스킬 원표) | CatBoost(이 노트북) | 범위 | 비고 |
|---|---|---|---|---|
| 학습 속도 | learning_rate | `learning_rate` | 0.01~0.1 (log) | |
| 트리 복잡도 | num_leaves | `depth` | 4~10 | CatBoost는 대칭 트리라 잎 개수 대신 깊이로 복잡도 조절 |
| 리프 최소 샘플 | min_data_in_leaf | `min_data_in_leaf` | 20~200 | 시계열 노이즈 대응 핵심(스킬 표와 동일) |
| 피처 샘플링 | feature_fraction | `rsm` | 0.6~1.0 | 각 분할에서 쓸 피처 비율 |
| 행 샘플링 | bagging_fraction | `subsample`(+`bootstrap_type="Bernoulli"`) | 0.6~1.0 | CatBoost 기본 부트스트랩(Bayesian)은 subsample 미지원이라 Bernoulli로 변경 필요 |
| L2 정규화 | lambda_l2 | `l2_leaf_reg` | 1~30 (log) | |

`iterations`는 스킬 1절 원칙대로 튜닝하지 않고 early stopping(100 rounds)으로 결정한다. trial 중에는 `iterations=1500`으로 상한만 낮게 잡아(탐색 속도), 최종 확정 파라미터로 재검증할 때 다시 넉넉하게 둔다.

**실행 시간 안내**: trial 1개마다 3-fold를 전부 학습한다. `N_TRIALS`는 아래에 상수로 뒀으니, 처음엔 작게(예: 20) 돌려보고 시간을 가늠한 뒤 늘리는 걸 추천한다. Optuna study는 `data/processed/optuna_catboost_pooled.db`(sqlite, git 제외)에 저장되어 **같은 기기에서는 나중에 이어서 탐색 가능**하다(`model-tuning` 스킬 5절).

In [ ]:
N_TRIALS = 30  # 시간 여유에 따라 조절 (trial 1개 = 3-fold 학습, 수십 초~수 분/trial 예상)


def objective(trial):
    params = {
        "iterations": 1500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "bootstrap_type": "Bernoulli",
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
    }
    mean_score, _ = cv_score(params)
    return mean_score


STUDY_DB = PROCESSED_DIR / "optuna_catboost_pooled.db"
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    study_name="catboost_pooled_v2",
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
)

n_done = len(study.trials)
study.optimize(objective, n_trials=N_TRIALS)
print(f"이번 실행에서 {len(study.trials) - n_done}개 trial 추가, 누적 {len(study.trials)}개")
print("최고 3-fold 평균 Score:", round(study.best_value, 4))
print("최적 파라미터:", study.best_params)

이번 실행에서 5개 trial 추가, 누적 5개
최고 3-fold 평균 Score: 0.591
최적 파라미터: {'learning_rate': 0.028580510658069373, 'depth': 9, 'l2_leaf_reg': 1.9721610970573997, 'min_data_in_leaf': 113, 'subsample': 0.836965827544817, 'rsm': 0.6185801650879991}


## 6. 탐색 결과 확인

상위 trial들의 파라미터 분포를 본다 — `model-tuning` 스킬 3절: "1차 coarse 탐색 → 상위 trial 분포 확인 → 필요하면 범위를 좁혀 2차 탐색." 상위 trial들이 특정 값 근처에 몰려 있으면 그 근방으로 범위를 좁혀 `N_TRIALS`를 늘려 재탐색하는 것도 방법이다.

In [ ]:
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
param_cols = [c for c in trials_df.columns if c.startswith("params_")]
trials_df[["number", "value", *param_cols]].head(10)

,number,value,params_depth,params_l2_leaf_reg,params_learning_rate,params_min_data_in_leaf,params_rsm,params_subsample
4,4,0.591035,9,1.972161,0.028581,113,0.618580,0.836966
3,3,0.590737,6,8.012738,0.027036,45,0.746545,0.716858
0,0,0.590279,10,12.057126,0.023689,128,0.662398,0.662407
2,2,0.589652,5,1.855998,0.067990,53,0.809903,0.721697
1,1,0.588405,10,7.725378,0.011431,148,0.987964,0.608234


## 7. seed 안정성 확인

`model-tuning` 스킬 3절: "최적 파라미터를 seed 3개로 재검증." 최적 파라미터가 특정 seed에만 우연히 맞은 건 아닌지 확인한다. 이번엔 `iterations`을 2000으로 다시 넉넉하게 둔다(트리 개수는 early stopping이 알아서 정하므로 상한만 여유 있게).

In [ ]:
BEST_PARAMS = {
    "iterations": 2000,
    "learning_rate": study.best_params["learning_rate"],
    "depth": study.best_params["depth"],
    "l2_leaf_reg": study.best_params["l2_leaf_reg"],
    "min_data_in_leaf": study.best_params["min_data_in_leaf"],
    "bootstrap_type": "Bernoulli",
    "subsample": study.best_params["subsample"],
    "rsm": study.best_params["rsm"],
}

seed_means = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score(BEST_PARAMS, seed=seed, verbose=False)
    seed_means.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

print(f"\nseed 3개 평균: {np.mean(seed_means):.4f}, 표준편차: {np.std(seed_means):.4f}")
print("(표준편차가 fold 간 점수 차이보다 훨씬 작으면 특정 seed 운이 아니라 안정적인 개선으로 판단)")

seed=42: 3-fold 평균 Score=0.5910 (fold별 [np.float64(0.5681), np.float64(0.5943), np.float64(0.6107)])
seed=7: 3-fold 평균 Score=0.5899 (fold별 [np.float64(0.5687), np.float64(0.5934), np.float64(0.6076)])
seed=2024: 3-fold 평균 Score=0.5876 (fold별 [np.float64(0.5643), np.float64(0.5888), np.float64(0.6098)])

seed 3개 평균: 0.5895, 표준편차: 0.0014
(표준편차가 fold 간 점수 차이보다 훨씬 작으면 특정 seed 운이 아니라 안정적인 개선으로 판단)


## 8. 튜닝 전후 비교 및 최종 파라미터

같은 3-fold CV 구조에서 기본값과 튜닝값을 나란히 놓고 본다. `model-tuning` 스킬 1절: "개선이 fold 표준편차 이내면 '튜닝 효과 미미'로 정직하게 기록" — 개선폭이 크지 않다고 실망할 필요 없이, 그 자체로 유효한 결론이다.

In [ ]:
comparison_df = pd.DataFrame({
    "구분": ["튜닝 전(기본값)", "튜닝 후(최적 파라미터, seed 평균)"],
    "3-fold 평균 Score": [baseline_mean, float(np.mean(seed_means))],
})
comparison_df["개선폭"] = comparison_df["3-fold 평균 Score"] - baseline_mean
comparison_df

,구분,3-fold 평균 Score,개선폭
0,튜닝 전(기본값),0.590835,0.000000
1,"튜닝 후(최적 파라미터, seed 평균)",0.589510,-0.001325


## 요약

- `model-tuning`/`timeseries-validation` 스킬 기준, 확장 윈도우 3-fold CV로 바꿔 단일 fold 과적합 위험을 줄였다
- CatBoost 파라미터를 LightGBM 스킬 표에 대응시켜 탐색 범위를 설계했다(`learning_rate`/`depth`/`min_data_in_leaf`/`rsm`/`subsample`/`l2_leaf_reg`)
- Optuna(TPE)로 `N_TRIALS`만큼 탐색, sqlite에 저장해 이어서 탐색 가능하게 했다
- 최적 파라미터를 seed 3개로 재검증해 안정성을 확인했다
- 튜닝 전후를 같은 CV 구조에서 비교했다(위 `comparison_df`)

**다음 할 일**:
1. 이 노트북을 실행해서 `N_TRIALS=30`으로 1차 탐색 결과를 확인 (시간이 오래 걸리면 `N_TRIALS`를 줄여서 먼저 테스트 추천)
2. 상위 trial 파라미터가 특정 범위에 몰려 있으면 그 근방으로 좁혀 2차 탐색(선택)
3. 결과를 `reports/05_tuning.md`에 정리
4. 개선이 확인되면 `BEST_PARAMS`를 `ensemble-final` 단계(`train.ipynb`)에 하드코딩해 전체 데이터로 재학습